In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [2]:
load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}")

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database()"))
    print(result.scalar())

olist


In [3]:
raw_dir = Path("..") / "data" / "raw"

files = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "product_category_name_translation",
}

with engine.begin() as conn:
    for filename, table in files.items():
        conn.execute(text(f"TRUNCATE TABLE staging.{table}"))
        df = pd.read_csv(raw_dir / filename, dtype=str, keep_default_na=False)
        df.to_sql(table, conn, schema="staging", if_exists="append", index=False)
        print(f"{table}: {len(df)} rows loaded")

customers: 99441 rows loaded
geolocation: 1000163 rows loaded
orders: 99441 rows loaded
order_items: 112650 rows loaded
order_payments: 103886 rows loaded
order_reviews: 99224 rows loaded
products: 32951 rows loaded
sellers: 3095 rows loaded
product_category_name_translation: 71 rows loaded


In [4]:
tables = list(files.values())

with engine.connect() as conn:
    for table in tables:
        n = conn.execute(text(f"SELECT COUNT(*) FROM staging.{table}")).scalar()
        print(f"{table}: {n} rows in database")

customers: 99441 rows in database
geolocation: 1000163 rows in database
orders: 99441 rows in database
order_items: 112650 rows in database
order_payments: 103886 rows in database
order_reviews: 99224 rows in database
products: 32951 rows in database
sellers: 3095 rows in database
product_category_name_translation: 71 rows in database
